# 🗄️ Vector Database Fundamentals

**Day 6 — Notebook 1 of 4 | Estimated Time: 25 minutes**

---

## What You'll Learn
- Why brute-force similarity search doesn't scale
- What a vector database is and how it differs from a traditional database
- How HNSW (Hierarchical Navigable Small World) indexing works conceptually
- The trade-off between approximate and exact nearest neighbour search
- An overview of the vector database landscape

---

## Setup

In [ ]:
%pip install -U -q "google-genai>=1.0.0" numpy chromadb

In [ ]:
import sys, os, time
import numpy as np
import matplotlib.pyplot as plt

repo_root = os.path.abspath(os.path.join(os.getcwd(), "../../.."))
if repo_root not in sys.path:
    sys.path.append(repo_root)

from helpers.auth import get_api_key
from google import genai
from google.genai import types

client = genai.Client(api_key=get_api_key())
EMBEDDING_MODEL = "text-embedding-004"
print("✅ Ready!")

---

## 1. The Scaling Problem with Brute-Force Search

In Day 5, we built a simple search index using numpy:

```python
scores = doc_vectors @ query_vector   # Compare query against ALL documents
```

This works fine for small corpora, but it **doesn't scale**:

| Documents | Dimensions | Comparisons per query | Approximate time |
|-----------|------------|----------------------|------------------|
| 1,000 | 768 | 768,000 ops | < 1ms |
| 100,000 | 768 | 76,800,000 ops | ~50ms |
| 10,000,000 | 768 | 7,680,000,000 ops | ~5 seconds |
| 1,000,000,000 | 768 | 768,000,000,000 ops | ~8 minutes |

For production systems — millions of documents, thousands of queries per second — this is completely impractical.

In [ ]:
# Demonstrate brute-force scaling problem
def brute_force_search_time(n_docs: int, n_dims: int = 768) -> float:
    """Measure time for brute-force nearest neighbour search."""
    # Create random vectors (simulating embeddings)
    vectors = np.random.randn(n_docs, n_dims).astype(np.float32)
    query = np.random.randn(n_dims).astype(np.float32)
    
    # Normalise
    vectors = vectors / np.linalg.norm(vectors, axis=1, keepdims=True)
    query = query / np.linalg.norm(query)
    
    start = time.perf_counter()
    scores = vectors @ query  # Brute-force
    _ = np.argmax(scores)     # Find the best match
    return (time.perf_counter() - start) * 1000  # ms


sizes = [1_000, 10_000, 100_000, 500_000]
times = []

print("Brute-force search scaling:\n")
for n in sizes:
    t = brute_force_search_time(n)
    times.append(t)
    print(f"  {n:>10,} docs → {t:>8.2f} ms")

print("\n⚠️  Notice how time grows linearly with document count.")

---

## 2. What Is a Vector Database?

A **vector database** is a database purpose-built for storing and searching embedding vectors. It solves the scaling problem using **Approximate Nearest Neighbour (ANN)** algorithms.

### Traditional DB vs Vector DB

| Feature | Traditional Database | Vector Database |
|---------|---------------------|------------------|
| **Storage** | Rows and columns (structured) | Vectors + metadata |
| **Query** | SQL: exact match, range, join | ANN: nearest neighbours |
| **Index type** | B-tree, hash index | HNSW, IVF, LSH |
| **Similarity** | Not applicable | Cosine, dot product, Euclidean |
| **Scale** | Billions of rows (exact) | Billions of vectors (approximate) |

Key insight: **you trade a tiny amount of accuracy for a massive speed gain**.

---

## 3. How HNSW Works (Conceptually)

**HNSW (Hierarchical Navigable Small World)** is the most widely used ANN algorithm. It's used by ChromaDB, Pinecone, Weaviate, and others.

The intuition:

```
Layer 2 (few nodes, long-range connections — "highway")
   A ─────────────────── E

Layer 1 (more nodes, medium connections — "main roads")
   A ── B ──── D ── E

Layer 0 (all nodes, short connections — "local streets")
   A─B─C─D─E─F─G─H─I─J
```

**Search algorithm:**
1. Start at the top layer — take big jumps to get close to the target
2. Move down layers — take smaller steps to refine
3. At the bottom layer — fine-tune to find the nearest neighbours

This turns an O(N) linear scan into O(log N) — orders of magnitude faster.

In [ ]:
# HNSW key parameters — understanding them helps tune performance
hnsw_params = {
    "M": {
        "default": 16,
        "description": "Max connections per node per layer",
        "low": "Faster build, lower accuracy, less memory",
        "high": "Slower build, higher accuracy, more memory",
    },
    "ef_construction": {
        "default": 200,
        "description": "Size of candidate list during index build",
        "low": "Faster indexing, lower recall",
        "high": "Slower indexing, higher recall",
    },
    "ef_search": {
        "default": 100,
        "description": "Size of candidate list during search",
        "low": "Faster queries, lower recall",
        "high": "Slower queries, higher recall",
    },
}

print("HNSW Tuning Parameters:\n")
for param, info in hnsw_params.items():
    print(f"  {param} (default: {info['default']})")
    print(f"    {info['description']}")
    print(f"    Low  → {info['low']}")
    print(f"    High → {info['high']}")
    print()

---

## 4. ANN vs Exact Search: The Accuracy Trade-off

ANN returns **approximate** results — sometimes the true nearest neighbour is missed. Let's measure this.

In [ ]:
import chromadb

# Build a ChromaDB collection (uses HNSW internally)
chroma_client = chromadb.Client()  # In-memory
collection = chroma_client.create_collection(
    name="ann_test",
    metadata={"hnsw:space": "cosine"}
)

# Add 1000 random vectors
np.random.seed(42)
n = 1000
dims = 768
vecs = np.random.randn(n, dims).astype(np.float32)
vecs = vecs / np.linalg.norm(vecs, axis=1, keepdims=True)

collection.add(
    ids=[str(i) for i in range(n)],
    embeddings=vecs.tolist(),
)

# Run a query
query_vec = np.random.randn(dims).astype(np.float32)
query_vec = query_vec / np.linalg.norm(query_vec)

# ANN search via ChromaDB
start = time.perf_counter()
ann_results = collection.query(query_embeddings=[query_vec.tolist()], n_results=5)
ann_time = (time.perf_counter() - start) * 1000
ann_ids = [int(x) for x in ann_results["ids"][0]]

# Exact search via numpy
start = time.perf_counter()
exact_scores = vecs @ query_vec
exact_ids = np.argsort(exact_scores)[::-1][:5].tolist()
exact_time = (time.perf_counter() - start) * 1000

# Compare
overlap = len(set(ann_ids) & set(exact_ids))
print(f"ANN (ChromaDB HNSW) top-5: {ann_ids}   [{ann_time:.2f}ms]")
print(f"Exact (numpy) top-5:       {exact_ids}  [{exact_time:.2f}ms]")
print(f"Overlap: {overlap}/5 ({overlap/5:.0%} recall)")
print(f"\nSpeedup: {exact_time/ann_time:.1f}x (ANN is faster at scale)")

# Cleanup
chroma_client.delete_collection("ann_test")

---

## 5. The Vector Database Landscape

There are many vector databases available. Here's how the main options compare:

| Database | Type | Best For | Notes |
|----------|------|----------|-------|
| **ChromaDB** | Open-source | Development, local use | Simple API, no server needed, we use this |
| **Pinecone** | Managed cloud | Production at scale | Fully managed, pay-per-use |
| **Weaviate** | Open-source / cloud | Hybrid search + GraphQL | Built-in ML models |
| **Qdrant** | Open-source / cloud | High-performance production | Rust-based, very fast |
| **pgvector** | PostgreSQL extension | Adding vectors to existing Postgres | Great if you already use Postgres |
| **Vertex AI Vector Search** | Google Cloud managed | GCP-native production | Billion-scale, integrates with Vertex AI |

For this course we use **ChromaDB** — it runs locally, requires no API keys, and the concepts transfer directly to any other vector database.

---

## 6. What Vector Databases Store

A vector database record has four components:

```
┌─────────────────────────────────────────────────────────┐
│  id:         "doc-001"                                  │
│  vector:     [0.21, -0.45, 0.83, ...]  (768 floats)    │
│  document:   "Python is a high-level language..."       │
│  metadata:   {"source": "wiki", "category": "tech",     │
│               "date": "2024-01", "chunk_index": 0}      │
└─────────────────────────────────────────────────────────┘
```

- **ID** — unique identifier (you define it)
- **Vector** — the embedding (computed from the document text)
- **Document** — the original text (stored for retrieval)
- **Metadata** — structured fields for filtering (category, date, source, etc.)

In [ ]:
# A preview of what ChromaDB records look like — we'll go deep in Notebook 2
client_preview = chromadb.Client()
col = client_preview.create_collection("preview")

col.add(
    ids=["doc-001", "doc-002"],
    documents=[
        "Python is a high-level programming language.",
        "The Eiffel Tower is in Paris, France.",
    ],
    embeddings=[
        np.random.randn(768).tolist(),  # Normally you'd use real embeddings
        np.random.randn(768).tolist(),
    ],
    metadatas=[
        {"category": "tech", "source": "wikipedia"},
        {"category": "travel", "source": "wikipedia"},
    ],
)

print(f"Collection count: {col.count()}")
print("Records stored — see Notebook 2 for full ChromaDB operations.")
client_preview.delete_collection("preview")

---

## 🏋️ Exercise 1: Scaling Comparison

Extend the brute-force timing experiment and plot how search time scales with corpus size.

In [ ]:
# Exercise 1: Plot brute-force search time vs corpus size
# TODO:
# 1. Measure brute-force search time for sizes: 1k, 5k, 10k, 50k, 100k
# 2. Run each size 3 times and take the average
# 3. Plot the results (x=corpus size, y=time in ms)
# 4. Is the relationship linear? What does that tell you about scalability?

sizes = [1_000, 5_000, 10_000, 50_000, 100_000]
avg_times = []

for n in sizes:
    pass  # TODO: measure and average 3 runs

# TODO: Plot with matplotlib
# plt.plot(sizes, avg_times, marker='o')
# plt.xlabel("Corpus Size")
# plt.ylabel("Search Time (ms)")
# plt.title("Brute-Force Search Scaling")
# plt.show()

---

## 🏋️ Exercise 2: ANN Recall vs Exact

Measure the recall of ChromaDB's HNSW index at different `top_k` values.

In [ ]:
# Exercise 2: Measure ANN recall at different k values
# TODO:
# 1. Create a ChromaDB collection with 5,000 random 768-dim vectors
# 2. Run 10 random queries
# 3. For each query, compare ANN top-k with exact top-k (using numpy)
# 4. Calculate average recall@k for k in [1, 5, 10, 20]
# 5. Print a summary table: k | avg_recall

# Your code here

---

## Key Takeaways

| Concept | Summary |
|---------|----------|
| **Brute-force search** | O(N) — works for thousands, breaks at millions |
| **ANN search** | O(log N) — trades tiny accuracy loss for massive speed gain |
| **HNSW** | Graph-based index with hierarchical layers — the dominant ANN algorithm |
| **Vector DB record** | ID + vector + document text + metadata |
| **ChromaDB** | Lightweight open-source vector DB — runs in-memory or on disk |

---

## 📚 Further Reading

| Resource | Type | Link |
|----------|------|------|
| HNSW Algorithm | Blog | [pinecone.io/learn/series/faiss/hnsw](https://www.pinecone.io/learn/series/faiss/hnsw/) |
| Vector Database Landscape | Blog | [pinecone.io/learn/vector-database](https://www.pinecone.io/learn/vector-database/) |
| ChromaDB Docs | Docs | [docs.trychroma.com](https://docs.trychroma.com/) |

---

**Next up:** [02_Getting_Started_with_ChromaDB.ipynb](./02_Getting_Started_with_ChromaDB.ipynb)